# Docker 实战

> Dockerfile / compose / volume / network

## 为什么需要它

知道 `docker run` 可以启动一个容器之后，下一步是把整个项目放进容器。但真实项目很少只有一个容器——web 服务 + 数据库 + 缓存 + 后台任务……如果每次都要敲一长串 `docker run` 命令，还要手动保证：网络互通（web 怎么连到 db？）、启动顺序（db 先于 web 启动）、数据不丢（db 容器删了数据还在）——很快就变成运维噩梦。

所以 Docker 生态提供了四件工具：**Dockerfile**（定义镜像内容）、**Volume**（数据持久化）、**Network**（容器间通信）、**Compose**（一键管理多容器）。

## 是什么

| 工具 | 解决的问题 |
|------|-----------|
| **Dockerfile** | 镜像是怎么做出来的？——用文本文件描述构建步骤 |
| **Volume** | 容器删了数据怎么留？——把数据存在宿主机，和容器生命周期解耦 |
| **Network** | 多个容器之间怎么通信？——放进同一个 Docker 网络，容器名就是域名 |
| **Compose** | 多个容器怎么一键管理？——用一个 YAML 文件定义所有服务，一条命令全启停 |

> **YAML**：一种用缩进表示层级关系的配置格式，比 JSON 更接近人类阅读习惯。深入 → [03-编码与序列化/YAML与TOML.ipynb](../03-编码与序列化/YAML与TOML.ipynb)

## 怎么样

### Dockerfile

一个纯文本文件，每行一个指令。构建时每条指令生成一层（layer），Docker 会缓存没变的层，只重建变化的部分。

In [ ]:
# Dockerfile 示例 —— Python Web 应用

FROM python:3.12-slim          # 基础镜像
WORKDIR /app                   # 设置工作目录
COPY requirements.txt .        # 先复制依赖文件
RUN pip install -r requirements.txt  # 安装依赖
COPY . .                       # 再复制项目代码
EXPOSE 8000                    # 声明端口
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]


关键指令：
- `FROM`：从哪个基础镜像开始（必选，必须是第一句）
- `COPY`：把宿主机文件拷贝进镜像
- `RUN`：构建时执行命令（如 `pip install`）
- `CMD`：容器启动时默认执行的命令
- `EXPOSE`：声明容器监听哪个端口（文档性质，真正映射靠 `docker run -p`）

**COPY 为什么分两次？** 如果先 COPY 整个项目再 RUN pip install，改一行代码也会导致 pip install 层缓存失效。而先 COPY requirements.txt 再 RUN pip install，之后 COPY . .，只要依赖文件不变，install 层就一直用缓存。

```dockerfile
COPY requirements.txt .   # 层 3：很少变
RUN pip install ...        # 层 4：很少变
COPY . .                   # 层 5：每次改代码都变
```

In [ ]:
# 构建镜像
docker build -t my-app:v1 .
# -t: 标签（名:版本）  .: 构建上下文路径


### Volume —— 数据持久化

容器内部的文件系统是临时的：删容器 = 丢数据。Volume 把宿主机的目录或 Docker 管理的卷挂载到容器里。

| 挂载方式 | 场景 |
|----------|------|
| `-v 宿主机路径:容器路径`（bind mount） | 开发时把代码目录挂进去，改代码即生效 |
| `-v 卷名:容器路径`（named volume） | Docker 管理，数据持久化，适合数据库 |
| `tmpfs` | 纯内存临时存储，容器停止就没了 |

In [ ]:
# 创建命名卷并挂载
docker volume create my-data
docker run -v my-data:/data my-app

# 开发时直接把本地代码目录挂进去
docker run -v $(pwd):/app my-app


### Network —— 容器间通信

Docker 默认每个容器有自己的隔离网络。把多个容器加入同一个自定义网络后，它们可以直接用**容器名**互相访问（Docker 内置了 DNS 解析）。

In [ ]:
# 创建网络
docker network create my-net

# 两个容器加入同一网络
docker run -d --name db --network my-net postgres:16
docker run -d --name web --network my-net -p 8080:8000 my-app

# web 容器里可以直接 ping db
# 数据库连接写成 db:5432 即可，不用记 IP


### Docker Compose —— 多容器编排

一个 `compose.yaml` 文件描述所有服务（容器）及其关系，一条命令全启动。这实现了 **Infrastructure as Code**（基础设施即代码）——环境配置变成了可版本管理的文本文件，在任何机器上 `docker compose up` 就能复现一样的环境。

In [ ]:
# compose.yaml
services:
  web:
    build: .
    ports:
      - "8080:8000"
    volumes:
      - .:/app
    depends_on:
      - db
    environment:
      - DATABASE_URL=postgresql://user:pass@db:5432/mydb

  db:
    image: postgres:16
    volumes:
      - pgdata:/var/lib/postgresql/data
    environment:
      - POSTGRES_PASSWORD=pass

volumes:
  pgdata:


In [ ]:
# Compose 常用命令
docker compose up -d           # 后台启动所有服务
docker compose ps              # 查看各服务状态
docker compose logs -f web     # 跟踪 web 服务日志
docker compose down            # 停止并删除容器、网络
docker compose down -v         # 连 volume 也删（数据没了！）


Compose 自动做了三件事：
1. 为项目创建独立 Docker 网络，所有服务自动加入
2. `depends_on` 控制启动顺序
3. 容器名 = hostname，服务间直接用 `db`、`web` 互相访问

## 相关笔记

- [01-管理工具/04-Docker概述.ipynb](01-管理工具/04-Docker概述.ipynb)——容器化概念 / image / container
- [09-基础设施/容器底层/镜像分层.ipynb](../09-基础设施/容器底层/镜像分层.ipynb)——镜像分层机制深入